# 내일 비가 올지 예측(Rain in Australia)
- 참고 : https://www.kaggle.com/prashant111/extensive-analysis-eda-fe-modelling
- 랜덤 포레스트(Random Forest)

# 관련 라이브러리 import

In [1]:
# 불필요한 경고 출력을 방지
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 분석/시각화/전처리/모델링에 필요한 라이브러리 로딩
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split

# 데이터 준비
- 아래 링크에서 weatherAUS.csv 파일을 다운로드 받기
#### https://www.kaggle.com/jsphyg/weather-dataset-rattle-package



## 데이터 읽기

In [ ]:
# weatherAUS.csv 파일 경로 지정 후 데이터프레임으로 읽기
data_path = "datas_ml/weatherAUS.csv"
df = pd.read_csv(data_path)
print(df.shape)  # (행 개수, 컬럼 개수) 확인

# EDA 수행

## 데이터 형상 파악하기

In [ ]:
# 데이터 상위 5개 행을 확인하여 컬럼 구성과 값 형태 파악
df.head()

In [ ]:
# 컬럼별 데이터 타입과 결측치 없는 개수(Non-Null Count) 확인
df.info()

In [ ]:
# 수치형 컬럼들의 기초 통계량(평균, 표준편차, 최소/최대 등) 확인
df.describe()

## 범주형 데이터 통계량 확인

In [ ]:
# 문자열(범주형) 타입의 컬럼 목록 추출
cat_columns = df.select_dtypes(exclude=np.number).columns
cat_columns

In [ ]:
# 범주형 컬럼들의 개수, 고유값 수, 최빈값, 빈도 확인
print(cat_columns)
df[cat_columns].describe()

## target(y) 데이터 확인

In [ ]:
# target(RainTomorrow)의 클래스별(Yes/No) 개수를 막대그래프로 확인 -> 데이터 불균형 여부 파악
sns.countplot(x="RainTomorrow", data=df)

## 상관 분석(Correlation Analysis) 수행해보기

### 수치형 데이터 확인

In [ ]:
# 숫자형(수치형) 컬럼 목록 추출
num_cols = df.select_dtypes(include=np.number).columns
num_cols

In [ ]:
# 수치형 컬럼만 따로 모은 임시 데이터프레임 생성 (상관관계 분석용)
temp_df = df[num_cols]
temp_df.head()

### 범주형 데이터를 수치형으로 변환
- RainToday, RainTomorrow 컬럼의 데이터를 수치형으로 변환
- 변환하여 수치형 데이터 프레임에 추가

In [ ]:
# RainToday, RainTomorrow(문자열 No/Yes)를 상관관계 분석에 포함시키기 위해 0/1 숫자로 변환하여 temp_df에 추가
temp_df['RainToday'] = df['RainToday'].map({'No': 0, 'Yes': 1}) #.astype(dtype='int')
temp_df['RainTomorrow'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1}) #.astype(dtype='int')

In [ ]:
# RainToday, RainTomorrow 컬럼이 추가되어 컬럼 수가 16 -> 18로 늘어난 것 확인
temp_df.shape

In [ ]:
# 수치형으로 변환된 RainToday, RainTomorrow가 포함된 데이터 확인
temp_df.head()

### 상관관계 계수 확인

In [ ]:
# 모든 수치형 피처(0/1로 변환한 RainToday, RainTomorrow 포함) 간의 상관계수 행렬 계산 후 히트맵으로 시각화
corr = temp_df.corr()
plt.figure(figsize=(25, 25));
sns.set(font_scale=1.5); # plot의 글자크기 설정
sns.heatmap(corr,
            vmax=0.8,
            #vmin=-1,
            linewidths=0.01,
            square=True,
            annot=True,
            fmt = '.2f', # annot의 출력 소숫점 자리 지정
            annot_kws={"size": 20},
            cmap='YlGnBu');
plt.title('Feature Correlation');

# 결측치 처리
## (null값) 확인하기

In [ ]:
# 컬럼별 결측치 비율을 오름차순으로 정렬해서 확인
df.isnull().mean().sort_values()

## categorical column 확인하기

In [ ]:
# (원본 df 기준으로) 범주형 컬럼 목록을 다시 추출
cat_cols = df.select_dtypes(exclude=np.number).columns
print(cat_columns)

## numerical column 확인하기

In [ ]:
# (원본 df 기준으로) 수치형 컬럼 목록을 다시 추출
num_cols = df.select_dtypes(include=np.number).columns
print(num_cols)

# 결측치 개수 확인하기

In [ ]:
# 범주형 컬럼들의 결측치 비율 확인
df[cat_cols].isnull().mean().sort_values()

In [ ]:
# 수치형 컬럼들의 결측치 비율 확인
df[num_cols].isnull().mean().sort_values()

## 결측치 채우기
### 수치형 컬럼 결측처리
- numerical value를 가진 column은 중위값(meidan)으로 채우기

In [ ]:
# Sunshine 컬럼의 결측치를 중앙값(median)으로 채움 (이상치에 덜 민감한 대표값)
#df['Sunshine'].fillna(df['Sunshine'].median())
df['Sunshine'] = df['Sunshine'].fillna(df['Sunshine'].median())

In [ ]:
# 나머지 수치형 컬럼들도 결측치가 있으면 각 컬럼의 중앙값으로 일괄 채움
for col in num_cols:
    if df[col].isnull().mean() > 0:
        col_median = df[col].median()
        df[col] = df[col].fillna(col_median)

### 범주형 컬럼 결측처리 
- null이 있는 범주형 컬럼 확인

In [ ]:
# 범주형 컬럼 중 결측치가 남아있는 컬럼과 그 비율을 확인
for col in cat_cols:
    if df[col].isnull().mean() > 0:
        print(col, (df[col].isnull().mean()))

In [ ]:
# WindGustDir 컬럼에서 가장 많이 등장한 값(최빈값) 확인
df['WindGustDir'].mode()[0]

In [ ]:
# 방법1
# categorical(범주형) 컬럼은 평균/중앙값을 구할 수 없으므로 최빈값(mode)으로 결측치를 채움
# RainTomorrow 는 target 이기 때문에 결측치 처리하지 않음.(일반적인 선택)
df['WindGustDir'] = df['WindGustDir'].fillna(df['WindGustDir'].mode()[0])  # 가장 흔한 풍향으로 채움
df['WindDir9am'] = df['WindDir9am'].fillna(df['WindDir9am'].mode()[0])    # 가장 흔한 풍향으로 채움
df['WindDir3pm'] = df['WindDir3pm'].fillna(df['WindDir3pm'].mode()[0])    # 가장 흔한 풍향으로 채움
df['RainToday'] = df['RainToday'].fillna(df['RainToday'].mode()[0])      # 가장 흔한 값(No/Yes)으로 채움

In [ ]:
# Index 객체를 list로 변환 (remove() 등 리스트 메서드 사용을 위해)
cat_cols= list(cat_cols)
cat_cols

In [ ]:
# RainTomorrow는 target이므로 결측치 처리 대상 리스트에서 제외
cat_cols.remove('RainTomorrow')
cat_cols

In [ ]:
# 방법2
# 방법1처럼 컬럼명을 일일이 나열하지 않고, cat_cols 리스트를 순회하면서
# 결측치가 있는 범주형 컬럼만 최빈값(mode)으로 자동 처리하는 방식
# (cat_cols에서 RainTomorrow는 이미 제거했으므로 target은 자동으로 제외됨)
# 방법1에서 이미 결측치를 채웠기 때문에 여기서는 실행하지 않고 주석 처리해 둠
# for col in cat_cols:
#     if df[col].isnull().mean() > 0:
#         df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# 결측치 처리 후 모든 컬럼이 Non-Null인지(단, RainTomorrow 제외) 최종 확인
df.info()

In [ ]:
# 컬럼별 결측치 개수 재확인 (RainTomorrow만 결측치가 남아있어야 정상)
df.isnull().sum()

## 필요 없는 컬럼 삭제 

In [ ]:
# 모델 학습에 사용하지 않을 Date 컬럼 삭제
df = df.drop(['Date'], axis=1)

In [ ]:
# Date 컬럼이 삭제된 것을 확인
df.head()

In [ ]:
# Date 삭제 후 컬럼별 결측치 비율 재확인
df.isnull().mean().sort_values()

## target의 null 값처리
- RainTomorrow는 target이기 때문에 nan은 drop으로 처리
- RainTomorrow 예측 값이므로 null 값 삭제함

In [ ]:
# 현재 데이터프레임 크기 확인 (RainTomorrow의 null 행 삭제 전)
df.shape

In [ ]:
# (위와 동일한 확인, 삭제 전 shape 재확인)
df.shape

In [ ]:
# RainTomorrow 컬럼에만 결측치가 남아있는지 최종 확인
df.isnull().sum()

In [ ]:
# RainTomorrow는 target이므로 값을 임의로 채우지 않고, 결측치가 있는 행 자체를 삭제
df = df.dropna(subset=['RainTomorrow'])
df.info()

In [ ]:
# 결측치 처리가 완료된 데이터 확인
df.head()

# Yes/No 값에 대한 변환

In [ ]:
# RainToday, RainTomorrow(문자열 No/Yes)를 모델이 학습할 수 있도록 0/1 숫자로 변환
df['RainToday'] = df['RainToday'].map({'No': 0, 'Yes': 1})
df['RainTomorrow'] = df['RainTomorrow'].map({'No': 0, 'Yes': 1})

print(df['RainToday'].dtype, df['RainTomorrow'].dtype)  # 변환 결과 dtype 확인

## categorical value에 one-hot encoding 적용하기

In [ ]:
# 원-핫 인코딩을 적용할 범주형 컬럼 목록 확인
cat_cols

In [ ]:
# 범주형 컬럼을 원-핫 인코딩(one-hot encoding) 적용
# pd.get_dummies()는 각 범주형 컬럼의 고유값마다 0/1로 이루어진 새 컬럼을 만들어줌
# 예) WindGustDir의 'W', 'N' 등 16개 방향값 -> WindGustDir_W, WindGustDir_N 등 16개 컬럼으로 분리됨
# Date는 이미 삭제했고, RainToday/RainTomorrow는 앞에서 0/1로 매핑했으므로 인코딩 대상에서 제외
df = pd.get_dummies(df, columns = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm'])

In [ ]:
# 원-핫 인코딩 적용 후 데이터 형태 확인
df.head()

In [ ]:
# 인코딩 후 전체 컬럼 수 확인 (범주형 컬럼들이 여러 개의 0/1 컬럼으로 늘어남)
df.shape

# ML을 위한 학습, 테스트 데이터 준비

In [44]:
# [문제] 특징 변수와 타겟 변수 분할 하기
X = df.drop('RainTomorrow', axis=1)
y = df['RainTomorrow']

In [45]:
X.head()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,...,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
0,13.4,22.9,0.6,4.8,8.4,44.0,20.0,24.0,71.0,22.0,...,False,False,False,False,False,False,False,False,True,False
1,7.4,25.1,0.0,4.8,8.4,44.0,4.0,22.0,44.0,25.0,...,False,False,False,False,False,False,False,False,False,True
2,12.9,25.7,0.0,4.8,8.4,46.0,19.0,26.0,38.0,30.0,...,False,False,False,False,False,False,False,False,False,True
3,9.2,28.0,0.0,4.8,8.4,24.0,11.0,9.0,45.0,16.0,...,False,False,False,False,False,False,False,False,False,False
4,17.5,32.3,1.0,4.8,8.4,41.0,7.0,20.0,82.0,33.0,...,False,True,False,False,False,False,False,False,False,False


In [46]:
y[:5]

0    0
1    0
2    0
3    0
4    0
Name: RainTomorrow, dtype: int64

In [47]:
y.value_counts()

RainTomorrow
0    110316
1     31877
Name: count, dtype: int64

In [48]:
# [문제]  학습, 테스트 데이터 분할 하기
# 학습 : 테스트 = 7 : 3 
# 불균형 데이터 이므로 stratify = y 로 설정
# 재현을 위해 seed 값은 2026 설정


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify = y, random_state=6 )

In [49]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((99535, 114), (42658, 114), (99535,), (42658,))

# ML 모델 학습

## 결정 트리(Decision Tree)로 내일 비가 올지 안올지 예측해보기

In [50]:
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
# [문제] 결정 트리 모델 학습 및 예측 수행
dt_clf = DecisionTreeClassifier()
dt_clf.fit(X_train, y_train)

y_pred = dt_clf.predict(X_test)

In [51]:
y_pred

array([0, 1, 0, ..., 0, 0, 0], shape=(42658,))

In [52]:
# [문제] 결정 트리 정확도 출력
score = accuracy_score(y_test, y_pred)
print('결정 트리(Decision Tree) Accuracy :',score)

결정 트리(Decision Tree) Accuracy : 0.7903792957944582


### [문제] 정확도, 정밀도, 재현율, f1-스코아를 값을 확인해 보세요.

In [53]:
from sklearn.metrics import classification_report
# [문제] classification_report() 결과 출력

# y_test: 실제 라벨, y_pred: 모델 예측 라벨
print("정확도:", accuracy_score(y_test, y_pred))
print("\n[정밀도, 재현율, F1-score 요약 보고서]")
print(classification_report(y_test, y_pred))

정확도: 0.7903792957944582

[정밀도, 재현율, F1-score 요약 보고서]
              precision    recall  f1-score   support

           0       0.87      0.86      0.86     33095
           1       0.53      0.55      0.54      9563

    accuracy                           0.79     42658
   macro avg       0.70      0.70      0.70     42658
weighted avg       0.79      0.79      0.79     42658



In [55]:
# [문제] roc_auc_score 출력, 소숫점 첫째자리까지만 출력
from sklearn.metrics import roc_auc_score

print(round(roc_auc_score(y_test, y_pred),1))

0.7


## 랜덤 포레스트(Random Forest)로  
- 참고 : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
- DT default = 100개를 만들어서 학습을 시킴

In [56]:
from sklearn.ensemble import RandomForestClassifier
# [문제] RandomForestClassifier 분류 모델 학습 및 예측 수행

rf_clf = RandomForestClassifier()

rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)

In [57]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(42658,))

In [59]:
# 정확도 평가
score = accuracy_score(y_test, y_pred)
print('랜덤 포레스트(Random Forest) Accuracy :', score)

랜덤 포레스트(Random Forest) Accuracy : 0.8589010267710628


In [ ]:
from sklearn.metrics import classification_report
# classification_report() 결과 출력
print("\n[정밀도, 재현율, F1-score 요약 보고서]")
print(classification_report(y_test, y_pred))


[정밀도, 재현율, F1-score 요약 보고서]
              precision    recall  f1-score   support

           0       0.87      0.96      0.91     33095
           1       0.78      0.51      0.62      9563

    accuracy                           0.86     42658
   macro avg       0.83      0.74      0.77     42658
weighted avg       0.85      0.86      0.85     42658



In [61]:
# [문제] roc_auc_score 출력, 소숫점 둘째자리까지만 출력
from sklearn.metrics import roc_auc_score

print(round(roc_auc_score(y_test, y_pred),2))

0.74


# XGBoost 분류기

In [63]:
from xgboost import XGBClassifier
# [문제] xgboost 학습 수행 및 예측

xgb_clf = XGBClassifier()
xgb_clf.fit(X_train, y_train)
y_pred = xgb_clf.predict(X_test)


In [64]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(42658,))

In [66]:
from sklearn.metrics import classification_report
# [문제] classification_report() 결과 출력
# y_test: 실제 라벨, y_pred: 모델 예측 라벨
print("정확도:", accuracy_score(y_test, y_pred))
print("\n[정밀도, 재현율, F1-score 요약 보고서]")
print(classification_report(y_test, y_pred))

정확도: 0.8608467344929439

[정밀도, 재현율, F1-score 요약 보고서]
              precision    recall  f1-score   support

           0       0.88      0.95      0.91     33095
           1       0.75      0.57      0.65      9563

    accuracy                           0.86     42658
   macro avg       0.82      0.76      0.78     42658
weighted avg       0.85      0.86      0.85     42658



In [67]:
# [문제] roc_auc_score 출력, 소숫점 둘째자리까지만 출력
from sklearn.metrics import roc_auc_score

print(round(roc_auc_score(y_test, y_pred),2))

0.76
